In [1]:
import json
import os

import numpy as np
import prettytable as pt
import pandas as pd
import pyarrow.parquet as pq
from scipy.optimize import curve_fit
from scipy.integrate import quad

import hist
# import matplotlib.pyplot as plt
# from sklearn.metrics import roc_curve, roc_auc_score
# from scipy.integrate import trapezoid

import eos_utils as eos

# Workspace packages
from HHtobbyy.event_discrimination.DFDataset import DFDataset
from HHtobbyy.workspace_utils import match_sample
from HHtobbyy.event_discrimination.models import map_model_to_Model


dfdataset_config = "/eos/uscms/store/group/lpcdihiggsboost/tsievert/HiggsDNA_parquet/DFDatasets/vManosv6/24_Manos_2026-04-20_12-49-04/dataset_config.json"
dfdataset = DFDataset(dfdataset_config)
# model, model_config = "MLP", {"output_dirpath": "/uscms/home/tsievert/nobackup/XHYbbgg/Model_Outputs/ManosMLPv6/2026-04-20_13-59-41", "activation_func": "ELU"}
# model = map_model_to_Model(model)(dfdataset, model_config)

/store/group/lpcdihiggsboost/tsievert/HiggsDNA_parquet/DFDatasets/vManosv6/24_Manos_2026-04-20_12-49-04/dataset_config.json


In [2]:
cats = {
    # "boosted": {
    #     "is_boosted": 1
    # },
    # "vbfhh_cat1": {
    #     "AUX_DVBFHH": 0.774,  # >
    # },
    "cat1": {
        "AUX_DggFHH": 0.9,  # >
        "AUX_DnonRes": 0.,  # <
        "AUX_DRes": 0.,  # <
    },
    "cat1copy": {
        "AUX_DggFHH": 0.99292177413867067,  # >
        "AUX_DnonRes": 0.42667485697897478,  # <
        "AUX_DRes": 0.90612735576941672,  # <
    },
    "cat2": {
        "AUX_DggFHH": 0.68115907008322718,  # >
        "AUX_DnonRes": 0.00064209200655934,  # <
        "AUX_DRes": 0.96341029269140221,  # <
    },
    "cat3" : {
        "AUX_DggFHH": 0.87036774207656309,  # >
        "AUX_DnonRes": 0.00221775294750575,  # <
        "AUX_DRes": 0.84544492824676509,  # <
    }
}

In [31]:
filters = [[(col, '>' if col.endswith('HH') else '<', cut) for col, cut in cats['cat1'].items()]]
filters

[[('AUX_DggFHH', '>', 0.9), ('AUX_DnonRes', '<', 0.0), ('AUX_DRes', '<', 0.0)]]

In [34]:
filepaths = dfdataset.get_test_filepaths(0, regex='')
filepaths

{'test': ['root://cmseos.fnal.gov//store/group/lpcdihiggsboost/tsievert/HiggsDNA_parquet/DFDatasets/vManosv6/24_Manos_2026-04-20_12-49-04/Run2_2016postVFP/data/DataF_2016postVFP_NOTAG_preprocessed_test0.parquet',
  'root://cmseos.fnal.gov//store/group/lpcdihiggsboost/tsievert/HiggsDNA_parquet/DFDatasets/vManosv6/24_Manos_2026-04-20_12-49-04/Run2_2016postVFP/data/DataG_2016postVFP_NOTAG_preprocessed_test0.parquet',
  'root://cmseos.fnal.gov//store/group/lpcdihiggsboost/tsievert/HiggsDNA_parquet/DFDatasets/vManosv6/24_Manos_2026-04-20_12-49-04/Run2_2016postVFP/data/DataH_2016postVFP_NOTAG_preprocessed_test0.parquet',
  'root://cmseos.fnal.gov//store/group/lpcdihiggsboost/tsievert/HiggsDNA_parquet/DFDatasets/vManosv6/24_Manos_2026-04-20_12-49-04/Run2_2016postVFP/sim/DDQCDGJets/DDQCDGJET_preprocessed_test0.parquet',
  'root://cmseos.fnal.gov//store/group/lpcdihiggsboost/tsievert/HiggsDNA_parquet/DFDatasets/vManosv6/24_Manos_2026-04-20_12-49-04/Run2_2016postVFP/sim/DDQCDGJets/GGJets_MGG-80_

In [3]:
# filepaths = dfdataset.get_test_filepaths(0, regex='')
filepaths = {'test': ['root://cmseos.fnal.gov//store/group/lpcdihiggsboost/tsievert/HiggsDNA_parquet/DFDatasets/vManosv6/24_Manos_2026-04-20_12-49-04/Run3_2024/sim/GluGlutoHH_kl-1p00_kt-1p00_c2-0p00/nominal/NOTAG_preprocessed_test0.parquet']}
test_pre = None
filters=None
for filepath in filepaths['test']:
    eos_filepath = eos.load_file_eos(filepath)
    table = pq.read_table(eos_filepath, filters=filters)
    if test_pre is None: test_pre = table.to_pandas()
    else: test_pre = pd.concat([test_pre, table.to_pandas()])
    eos.delete_lockfile(eos_filepath)

dipho_mass_window = np.logical_and(test_pre['AUX_mass'].gt(100).to_numpy(), test_pre['AUX_mass'].lt(180).to_numpy())
pho_mva_cut = test_pre['AUX_pass_mva-0.7'].gt(0).to_numpy()
snt_cuts = np.logical_and(dipho_mass_window, pho_mva_cut)
test = test_pre.loc[snt_cuts]



# expected_eras = {
#     # sim
#     'Run2_2016postVFP/sim', 'Run2_2016preVFP/sim', 'Run2_2017/sim', 'Run2_2018/sim',
#     'Run3_2022/sim/postEE', 'Run3_2022/sim/preEE', 'Run3_2023/sim/postBPix', 'Run3_2023/sim/preBPix', 'Run3_2024/sim', 'Run3_2025/sim',
#     # data
#     'Run2_2016postVFP/data', 'Run2_2016preVFP/data', 'Run2_2017/data', 'Run2_2018/data',
#     'Run3_2022/data', 'Run3_2023/data', 'Run3_2024/data', 'Run3_2025/data'
# }
# set_of_unique_sample_eras = set(pd.unique(test_pre['AUX_sample_era']).tolist())
# assert expected_eras == set_of_unique_sample_eras, f"sample eras do not match expected (either missing eras or names changed): \n {expected_eras} \n vs. \n {set_of_unique_sample_eras}"
# for sample_era in pd.unique(test_pre['AUX_sample_era']):
#     set_of_unique_sample_names = set(pd.unique(test_pre.loc[test_pre['AUX_sample_era'].eq(sample_era), 'AUX_sample_name']).tolist())
#     if 'data' in sample_era:
#         expected_set = {'Data'}
#     elif 'sim' in sample_era:
#         expected_set = {
#             'DDQCDGJets', 'GGJets', 'TTGG', 
#             'GluGluHToGG', 'VBFHToGG', 'VHToGG', 'ttHToGG', 'bbHToGG',
#             'GluGluToHH_kl1p00_kt1p00_c20p00', 'VBFToHH_cv1p00_c2v1p00_c31p00'
#         }
#         if '201' in sample_era: expected_set.remove('bbHToGG')
#         elif '2022' in sample_era: expected_set.remove('VBFToHH_cv1p00_c2v1p00_c31p00')
#     else:
#         raise Exception(f"Improper naming for era, expected to either contain \'sim\' or \'data\': {sample_era}")
#     assert set_of_unique_sample_names == expected_set, f"{sample_era} - samples names do not match expected: \n {expected_set} \n vs. \n {set_of_unique_sample_names}"
#     # for sample_name in pd.unique(test_pre.loc[test_pre['AUX_sample_era'].eq(sample_era), 'AUX_sample_name']):
#     #     print('-'*60, '\n', sample_name)


# vbf23_mask = np.logical_and(
#     test['AUX_sample_era'].isin(['Run3_2023/sim/preBPix', 'Run3_2023/sim/postBPix']),
#     test['AUX_sample_name'].eq('VBFHH')
# )  # makeup for lack of 2022 VBFHH signal 
# test.loc[vbf23_mask, 'AUX_eventWeight'] = 1.78 * test.loc[vbf23_mask, 'AUX_eventWeight']


In [48]:
len(test)

65845

In [6]:
test.head()

,eta,lead_eta,lead_phi,lead_mvaID,sublead_eta,sublead_phi,sublead_mvaID,nonResReg_vbfpair_lead_bjet_eta,nonResReg_vbfpair_lead_bjet_phi,nonResReg_vbfpair_lead_bjet_bTagWPL,...,AUX_nonResReg_vbfpair_HHbbggCandidate_mass,AUX_nonResReg_vbfpair_max_nonbjet_btag,AUX_nonResReg_vbfpair_resolved_BDT_mask,AUX_pass_mva-0.7,AUX_label1D,AUX_eventWeightTrain,AUX_DnonRes,AUX_DRes,AUX_DggFHH,AUX_DVBFHH
0,0.848661,1.108154,-0.720947,0.721321,-0.086243,-1.927734,0.852596,-0.492065,1.846924,1.0,...,425.238557,0.007258,1.0,1.0,2,0.000060,0.707463,0.292536,1.057823e-06,1.824979e-07
1,-2.263550,-2.253906,1.472168,0.798535,-1.444092,2.763672,0.865144,-0.533691,-1.383789,1.0,...,496.108216,0.989624,0.0,1.0,2,0.000022,0.995248,0.004743,5.682278e-06,3.595901e-06
2,1.437560,1.255371,0.625244,0.530829,1.037109,2.167969,0.800104,1.078613,-1.808105,0.0,...,247.478440,0.066025,1.0,1.0,2,0.000076,0.002699,0.997301,1.110180e-08,2.029395e-09
3,-0.621169,0.378052,0.746094,0.832254,-1.576660,2.175293,0.815609,-0.250183,-1.517334,1.0,...,376.356250,0.998169,1.0,1.0,2,0.000035,0.999954,0.000046,2.741084e-08,7.489380e-08
4,0.260175,0.385132,1.870117,0.807224,-0.222931,1.885254,0.875100,-0.443237,-1.279053,1.0,...,1100.597238,0.974731,1.0,1.0,2,0.000097,0.047966,0.952034,1.454281e-08,1.942704e-09


In [5]:
test.iloc[0].to_numpy()

array([0.8486606620027043, 1.108154296875, -0.720947265625,
       0.7213212847709656, -0.08624267578125, -1.927734375,
       0.8525958061218262, -0.4920654296875, 1.846923828125, 1.0, 1.0,
       1.0, 1.0, 1.0, -1.380126953125, -2.8046875, 0.0, 0.0, 0.0, 0.0,
       0.0, 3.025667905807495, 3.2455379962921143, 2.5411415100097656,
       1.5630686283111572, 1.5630686283111572, 0.6693284829804875,
       0.5592007115301617, 0.5458831657192881, -2.56982421875,
       45.136348724365234, 0.0, 2.0, -9.0, -9.0, -1.8664371967315674,
       -0.23486328125, -9.0, -9.0, -9.0, -9.0, -9.0, -9.0, -9.0, -9.0,
       -9.0, -9.0, -9.0, -9.0, -9.0, -9.0, -9.0, -9.0, -9.0, -9.0, -9.0,
       -9.0, 0.4734447347459655, 0.30158160663322336, 0.3215926450858437,
       1.2212590634253129, 0.2271242372672357, 111.3119682157064, 7148.0,
       153.0, 0.0, 'GluGluToHH_kl1p00_kt1p00_c20p00', 'Run3_2024/sim',
       6.159692309037822e-06, 6.014707590401844e-05, 127.37071274797454,
       111.3119682157064, 425.2

In [56]:
test.loc[test['eta'].gt(0.49283) & test['eta'].lt(0.49284)].head()

,eta,lead_eta,lead_phi,lead_mvaID,sublead_eta,sublead_phi,sublead_mvaID,nonResReg_vbfpair_lead_bjet_eta,nonResReg_vbfpair_lead_bjet_phi,nonResReg_vbfpair_lead_bjet_bTagWPL,...,AUX_nonResReg_vbfpair_HHbbggCandidate_mass,AUX_nonResReg_vbfpair_max_nonbjet_btag,AUX_nonResReg_vbfpair_resolved_BDT_mask,AUX_pass_mva-0.7,AUX_label1D,AUX_eventWeightTrain,AUX_DnonRes,AUX_DRes,AUX_DggFHH,AUX_DVBFHH
41911,0.492832,0.550537,-1.294434,0.954421,0.105042,-2.602051,0.933844,1.428467,1.365234,1.0,...,301.042623,0.983887,1.0,1.0,2,0.000074,0.021187,0.978813,1.252812e-08,1.687811e-09


In [52]:
test.head()

,eta,lead_eta,lead_phi,lead_mvaID,sublead_eta,sublead_phi,sublead_mvaID,nonResReg_vbfpair_lead_bjet_eta,nonResReg_vbfpair_lead_bjet_phi,nonResReg_vbfpair_lead_bjet_bTagWPL,...,AUX_nonResReg_vbfpair_HHbbggCandidate_mass,AUX_nonResReg_vbfpair_max_nonbjet_btag,AUX_nonResReg_vbfpair_resolved_BDT_mask,AUX_pass_mva-0.7,AUX_label1D,AUX_eventWeightTrain,AUX_DnonRes,AUX_DRes,AUX_DggFHH,AUX_DVBFHH
0,0.848661,1.108154,-0.720947,0.721321,-0.086243,-1.927734,0.852596,-0.492065,1.846924,1.0,...,425.238557,0.007258,1.0,1.0,2,0.000060,0.707463,0.292536,1.057823e-06,1.824979e-07
1,-2.263550,-2.253906,1.472168,0.798535,-1.444092,2.763672,0.865144,-0.533691,-1.383789,1.0,...,496.108216,0.989624,0.0,1.0,2,0.000022,0.995248,0.004743,5.682278e-06,3.595901e-06
2,1.437560,1.255371,0.625244,0.530829,1.037109,2.167969,0.800104,1.078613,-1.808105,0.0,...,247.478440,0.066025,1.0,1.0,2,0.000076,0.002699,0.997301,1.110180e-08,2.029395e-09
3,-0.621169,0.378052,0.746094,0.832254,-1.576660,2.175293,0.815609,-0.250183,-1.517334,1.0,...,376.356250,0.998169,1.0,1.0,2,0.000035,0.999954,0.000046,2.741084e-08,7.489380e-08
4,0.260175,0.385132,1.870117,0.807224,-0.222931,1.885254,0.875100,-0.443237,-1.279053,1.0,...,1100.597238,0.974731,1.0,1.0,2,0.000097,0.047966,0.952034,1.454281e-08,1.942704e-09


In [49]:
test.loc[:, [col for col in test.columns if col.startswith('AUX')]].head()

,AUX_event,AUX_lumi,AUX_hash,AUX_sample_name,AUX_sample_era,AUX_weight,AUX_eventWeight,AUX_mass,AUX_nonResReg_vbfpair_dijet_mass_DNNreg,AUX_nonResReg_vbfpair_HHbbggCandidate_mass,AUX_nonResReg_vbfpair_max_nonbjet_btag,AUX_nonResReg_vbfpair_resolved_BDT_mask,AUX_pass_mva-0.7,AUX_label1D,AUX_eventWeightTrain,AUX_DnonRes,AUX_DRes,AUX_DggFHH,AUX_DVBFHH
0,7148.0,153.0,0.0,GluGluToHH_kl1p00_kt1p00_c20p00,Run3_2024/sim,0.000006,0.000060,127.370713,111.311968,425.238557,0.007258,1.0,1.0,2,0.000060,0.707463,0.292536,1.057823e-06,1.824979e-07
1,7154.0,153.0,1.0,GluGluToHH_kl1p00_kt1p00_c20p00,Run3_2024/sim,0.000002,0.000022,123.036786,111.180941,496.108216,0.989624,0.0,1.0,2,0.000022,0.995248,0.004743,5.682278e-06,3.595901e-06
2,7156.0,153.0,2.0,GluGluToHH_kl1p00_kt1p00_c20p00,Run3_2024/sim,0.000008,0.000076,123.981862,75.154049,247.478440,0.066025,1.0,1.0,2,0.000076,0.002699,0.997301,1.110180e-08,2.029395e-09
3,7162.0,153.0,3.0,GluGluToHH_kl1p00_kt1p00_c20p00,Run3_2024/sim,0.000004,0.000035,126.278114,138.788839,376.356250,0.998169,1.0,1.0,2,0.000035,0.999954,0.000046,2.741084e-08,7.489380e-08
4,7158.0,153.0,4.0,GluGluToHH_kl1p00_kt1p00_c20p00,Run3_2024/sim,0.000010,0.000097,125.125739,116.217368,1100.597238,0.974731,1.0,1.0,2,0.000097,0.047966,0.952034,1.454281e-08,1.942704e-09


In [46]:
manos_sample = pd.read_parquet('/uscms/home/tsievert/nobackup/XHYbbgg/v6_merged_scored_events.parquet', filters=[("year", "==", 2024), ("sample", "==", "GluGluToHH_kl-1p00_kt-1p00_c2-0p00")])

In [47]:
len(manos_sample)

65845

In [50]:
manos_sample.head()

,sample,year,lumi,event,run,mass,nonResReg_vbfpair_dijet_mass_DNNreg,is_boosted,y_proba,weight,weight_central,weight_interference,weight_nominal,rel_xsec_weight,weight_tot,score,is_nonRes_bkg_score,is_Res_bkg_score,is_ggHH_sig_score,is_VBFHH_sig_score
0,GluGluToHH_kl-1p00_kt-1p00_c2-0p00,2024,153,7148,1,127.370713,111.311968,False,-99.000000,0.000006,0.903983,0.0,0.029916,0.000061,0.000061,"[0.004668295, 0.025433194, 0.95904833, 0.01085...",0.004668,0.025433,0.959048,0.010850
1,GluGluToHH_kl-1p00_kt-1p00_c2-0p00,2024,153,7154,1,123.036786,111.180941,False,-99.000000,0.000002,0.325072,0.0,0.010758,0.000022,0.000022,"[0.00027772892, 0.0046289414, 0.9930479, 0.002...",0.000278,0.004629,0.993048,0.002046
2,GluGluToHH_kl-1p00_kt-1p00_c2-0p00,2024,153,7156,1,123.981862,75.154049,False,-99.000000,0.000008,1.139399,0.0,0.037707,0.000076,0.000076,"[0.12222364, 0.68658906, 0.18682529, 0.004361956]",0.122224,0.686589,0.186825,0.004362
3,GluGluToHH_kl-1p00_kt-1p00_c2-0p00,2024,153,7162,1,126.278114,138.788839,False,-99.000000,0.000004,0.529351,0.0,0.017518,0.000035,0.000035,"[0.00178588, 0.016526002, 0.97376525, 0.007922...",0.001786,0.016526,0.973765,0.007923
4,GluGluToHH_kl-1p00_kt-1p00_c2-0p00,2024,153,7158,1,125.125739,116.217368,True,0.970269,0.000010,1.453757,0.0,0.048110,0.000097,0.000097,"[0.00040912125, 0.008337724, 0.9800724, 0.0111...",0.000409,0.008338,0.980072,0.011181


In [4]:
#############################################################
# ASCii histogram for rapid plotting
def ascii_hist(x, bins=10, weights=None):
    N,X = np.histogram(x, bins=bins, weights=weights)
    width = 50
    nmax = np.max(N.max())
    for (xi, n) in zip(X,N):
        bar = '#'*int(n*width/nmax)
        xi = '{0: <8.4g}'.format(xi).ljust(10)
        print('{0}| {1}'.format(xi,bar))
def ascii_hist(x, bins=10, weights=None, fit=None):
    N,X = np.histogram(x, bins=bins, weights=weights)
    width = 50
    nmax = np.max([N.max(), fit.max()])
    if fit is None:
        for (xi, n) in zip(X,N):
            bar = '#'*int(n*width/nmax)
            xi = '{0: <8.4g}'.format(xi).ljust(10)
            print('{0}| {1}'.format(xi,bar))
    else:
        for (xi, n, fiti) in zip(X,N,fit):
            bar = '#'*int(n*width/nmax)
            if fiti > n: bar = bar + ' '*(int(fiti*width/nmax) - int(n*width/nmax)) + '+'
            else: bar = ''.join([bar[j] if j != int(fiti*width/nmax) else '+' for j in range(len(bar))])
            xi = '{0: <8.4g}'.format(xi).ljust(10)
            print('{0}| {1}'.format(xi,bar))

#############################################################
# Sideband fit functions for nonRes bkg estimaton
def exp_func(x, a, b):
    return a * np.exp(b * x)
def sd_hist(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float]):
    return hist.Hist(
        hist.axis.Regular(int((fit_bins[1]-fit_bins[0])//fit_bins[2]), fit_bins[0], fit_bins[1], name="var", growth=False, underflow=False, overflow=False), 
        storage='weight'
    ).fill(var=mass, weight=weight)
def exp_mass_fit(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float], sigma: bool=False):
    _hist_ = sd_hist(mass, weight, fit_bins)
    params, _ = curve_fit(
        exp_func, _hist_.axes.centers[0]-_hist_.axes.centers[0][0], _hist_.values(), p0=(_hist_.values()[0], -0.1), 
        sigma=np.where(_hist_.values() != 0, np.sqrt(_hist_.variances()), 0.76) if sigma else None
    )
    print(_hist_)
    ascii_hist(mass, bins=np.arange(fit_bins[0], fit_bins[1], fit_bins[2]), weights=weight, fit=exp_func(_hist_.axes.centers[0]-_hist_.axes.centers[0][0], a=params[0], b=params[1]))
    return _hist_, params
def est_yield(mass: np.ndarray, weight: np.ndarray, fit_bins: list[float], sr_masscut: list[float], sigma: bool=False):
    _hist_, fit_params = exp_mass_fit(mass, weight, fit_bins, sigma=sigma)
    return quad(exp_func, sr_masscut[0]-_hist_.axes.centers[0][0], sr_masscut[1]-_hist_.axes.centers[0][0], args=tuple(fit_params))[0] / fit_bins[2]

In [14]:
data_samples = [sample_name for sample_name in sorted(pd.unique(test['AUX_sample_name']).tolist()) if match_sample(sample_name, ['Data']) is not None]
sideband_samples = [sample_name for sample_name in sorted(pd.unique(test['AUX_sample_name']).tolist()) if match_sample(sample_name, ['GJet', 'TTG']) is not None]
non_sideband_samples = [sample_name for sample_name in sorted(pd.unique(test['AUX_sample_name']).tolist()) if sample_name not in data_samples+sideband_samples]
print(f"All samples: {sorted(pd.unique(test['AUX_sample_name']).tolist())}")
print(f"Data samples: {data_samples}")
print(f"nonRes MC samples: {sideband_samples}")
print(f"Res/Signal MC samples: {non_sideband_samples}")


# field_names = ['Category'] + non_sideband_samples + ['nonRes SB fit', 'data SB fit', 's/b with nonRes fit', 's/b with data fit']
# field_names = ['Category'] + non_sideband_samples + ['nonRes SB fit', 'data SB fit']
# field_names = ['Category'] + non_sideband_samples + sideband_samples + sideband_samples + ['nonRes SB fit', 'N Data SB', 'data SB fit']

field_names = ['Category'] + non_sideband_samples + sideband_samples + ['single-H', 'non-Res', 'N Data SB']
table = pt.PrettyTable(field_names=field_names, float_format=".5")

not_prev_cut_mask = {}
for i, (name, cuts) in enumerate(cats.items()):
    if i == 0: continue

    for eras in [['Run2_2016preVFP/sim', 'Run2_2016postVFP/sim', 'Run2_2016preVFP/data', 'Run2_2016postVFP/data'], ['Run2_2017/sim', 'Run2_2017/data'], ['Run2_2018/sim', 'Run2_2018/data'], ['Run3_2022/sim/preEE', 'Run3_2022/sim/postEE', 'Run3_2022/data'], ['Run3_2023/sim/preBPix', 'Run3_2023/sim/postBPix', 'Run3_2023/data'], ['Run3_2024/sim', 'Run3_2024/data'], ['Run3_2025/sim', 'Run3_2025/data'], ['tot']]:
        if 'tot' not in eras:
            era_mask = test['AUX_sample_era'].isin(eras); evtwt_factor = 1
            era_name = eras[-1].split('_')[-1].replace('/data', '').replace('postVFP', '')
        else:
            era_mask = np.ones(len(test), dtype=bool); evtwt_factor = 1
            era_name = 'tot'
        new_row = [name+' '+era_name]
        nonRes_sideband = pd.DataFrame({'mass': pd.Series(dtype='float'), 'weight_tot': pd.Series(dtype='float')})

        singleH_yield = 0.
        nonRes_yield = 0.

        for sample in non_sideband_samples+sideband_samples:

            sample_mask = np.logical_and(era_mask, test['AUX_sample_name'].eq(sample))
            if sample+era_name not in not_prev_cut_mask: not_prev_cut_mask[sample+era_name] = sample_mask

            pass_cut_mask = not_prev_cut_mask[sample+era_name]
            if 'boost' not in name:
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, np.logical_and(test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].gt(80), test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].lt(190))
                )
            for cut_name, cut in cuts.items():
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, test.loc[:, cut_name].gt(cut).to_numpy() 
                    if 'HH' in cut_name else (
                        test.loc[:, cut_name].lt(cut).to_numpy() 
                        if 'D' in cut_name else 
                        test.loc[:, cut_name].eq(cut).to_numpy()
                    )
                )
            pass_cut_sr_mask = np.logical_and(
                pass_cut_mask,
                np.logical_and(test.loc[:, 'AUX_mass'].gt(115.).to_numpy(), test.loc[:, 'AUX_mass'].lt(135.).to_numpy())
            )
            new_row.append((evtwt_factor * test.loc[pass_cut_sr_mask, 'AUX_eventWeight']).sum())

            if 'H' in sample and 'HH' not in sample: singleH_yield += new_row[-1]
            elif 'H' not in sample: nonRes_yield += new_row[-1]

            not_prev_cut_mask[sample+era_name] = np.logical_and(not_prev_cut_mask[sample+era_name], ~pass_cut_mask)

        new_row.append(singleH_yield); new_row.append(nonRes_yield)

        # for sb_sample_name, sb_sample_arr in zip(['nonRes', 'data'], [sideband_samples, data_samples]):
        #     sb_mask = np.zeros(len(test), dtype=bool)
        #     for sample in sb_sample_arr: sb_mask = np.logical_or(sb_mask, test['AUX_sample_name'].eq(sample))
        #     if sb_sample_name not in not_prev_cut_mask: not_prev_cut_mask[sb_sample_name] = sb_mask

        #     pass_cut_mask = not_prev_cut_mask[sb_sample_name]
        #     if i != 0:
        #         pass_cut_mask = np.logical_and(
        #             pass_cut_mask, np.logical_and(test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].gt(80), test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].lt(190))
        #         )
        #     for cut_name, cut in cuts.items():
        #         pass_cut_mask = np.logical_and(
        #             pass_cut_mask, test.loc[:, cut_name].gt(cut).to_numpy() 
        #             if 'HH' in cut_name else (
        #                 test.loc[:, cut_name].lt(cut).to_numpy() 
        #                 if 'D' in cut_name else 
        #                 test.loc[:, cut_name].eq(cut).to_numpy()
        #             )
        #         )
        #     pass_cut_sb_mask = np.logical_and(
        #         pass_cut_mask,
        #         np.logical_or(test.loc[:, 'AUX_mass'].lt(115.).to_numpy(), test.loc[:, 'AUX_mass'].gt(135.).to_numpy())
        #     )
        #     if sb_sample_name == 'data':
        #         sb_window_mask = np.logical_and(test.loc[:, 'AUX_mass'].gt(100.).to_numpy(), test.loc[:, 'AUX_mass'].lt(180.).to_numpy())
        #         new_row.append(np.sum(np.logical_and(pass_cut_sb_mask, sb_window_mask)))
        #     if np.sum(pass_cut_sb_mask) != 0:
        #         sb_est_yield = est_yield(test.loc[pass_cut_sb_mask, 'AUX_mass'], test.loc[pass_cut_sb_mask, 'AUX_eventWeight'], [100., 180., 5.], [115., 135.])
        #     else:
        #         sb_est_yield = 0
        #     new_row.append(sb_est_yield)

        for sb_sample_name, sb_sample_arr in zip(['data'], [data_samples]):
            sb_mask = np.logical_and(era_mask, test['AUX_sample_name'].isin(sb_sample_arr))
            if sb_sample_name+era_name not in not_prev_cut_mask: not_prev_cut_mask[sb_sample_name+era_name] = sb_mask

            pass_cut_mask = not_prev_cut_mask[sb_sample_name+era_name]
            if i != 0:
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, np.logical_and(test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].gt(80), test.loc[:, 'AUX_nonResReg_vbfpair_dijet_mass_DNNreg'].lt(190))
                )
            for cut_name, cut in cuts.items():
                pass_cut_mask = np.logical_and(
                    pass_cut_mask, test.loc[:, cut_name].gt(cut).to_numpy() 
                    if 'HH' in cut_name else (
                        test.loc[:, cut_name].lt(cut).to_numpy() 
                        if 'D' in cut_name else 
                        test.loc[:, cut_name].eq(cut).to_numpy()
                    )
                )
            pass_cut_sb_mask = np.logical_and(
                pass_cut_mask,
                np.logical_or(test.loc[:, 'AUX_mass'].lt(115.).to_numpy(), test.loc[:, 'AUX_mass'].gt(135.).to_numpy())
            )
            sb_window_mask = np.logical_and(test.loc[:, 'AUX_mass'].gt(100.).to_numpy(), test.loc[:, 'AUX_mass'].lt(180.).to_numpy())
            new_row.append(np.sum(np.logical_and(pass_cut_sb_mask, sb_window_mask)))

        # sum_singleH = new_row[1] + sum(new_row[4:11])
        # new_row.append(new_row[2] / (sum_singleH + new_row[11]))
        # new_row.append(new_row[2] / (sum_singleH + new_row[12]))

        table.add_row(new_row)

print(table)


All samples: ['DDQCDGJets', 'Data', 'GGJets', 'GluGluHToGG', 'GluGluToHH_kl1p00_kt1p00_c20p00', 'TTGG', 'VBFHToGG', 'VBFToHH_cv1p00_c2v1p00_c31p00', 'VHToGG', 'bbHToGG', 'ttHToGG']
Data samples: ['Data']
nonRes MC samples: ['DDQCDGJets', 'GGJets', 'TTGG']
Res/Signal MC samples: ['GluGluHToGG', 'GluGluToHH_kl1p00_kt1p00_c20p00', 'VBFHToGG', 'VBFToHH_cv1p00_c2v1p00_c31p00', 'VHToGG', 'bbHToGG', 'ttHToGG']
+-----------+-------------+---------------------------------+----------+-------------------------------+---------+---------+---------+------------+---------+---------+----------+---------+-----------+
|  Category | GluGluHToGG | GluGluToHH_kl1p00_kt1p00_c20p00 | VBFHToGG | VBFToHH_cv1p00_c2v1p00_c31p00 |  VHToGG | bbHToGG | ttHToGG | DDQCDGJets |  GGJets |   TTGG  | single-H | non-Res | N Data SB |
+-----------+-------------+---------------------------------+----------+-------------------------------+---------+---------+---------+------------+---------+---------+----------+---------+---

In [ ]:
manos_sample = pd.read_parquet('/uscms/home/tsievert/nobackup/XHYbbgg/v6_merged_scored_events.parquet')
manos_sample['lumievent'] = (manos_sample['lumi'].astype(int).astype(str) + manos_sample['event'].astype(int).astype(str)).astype(int)
manos_sample.keys()

In [ ]:
for sample in pd.unique(manos_sample['sample']):
    sample_mask = manos_sample['sample'].eq(sample)
    # for year in pd.unique(manos_sample['year']):
    #     year_mask = manos_sample['year'].eq(year)
    #     print(sample, ' ', year, ' ', len(manos_sample.loc[np.logical_and(sample_mask, year_mask)]))
    print('num events of ', sample, ' ', 'all years', ' ', len(manos_sample.loc[sample_mask]))
    print('-'*60)

In [ ]:
for sample in pd.unique(test['AUX_sample_name']):
    sample_mask = test['AUX_sample_name'].eq(sample)
    # for year, year_tags in zip(['2016', '2017', '2018', '2022', '2023', '2024', '2025'], [['Run2_2016preVFP/sim', 'Run2_2016postVFP/sim', 'Run2_2016preVFP/data', 'Run2_2016postVFP/data'], ['Run2_2017/sim', 'Run2_2017/data'], ['Run2_2018/sim', 'Run2_2018/data'], ['Run3_2022/sim/preEE', 'Run3_2022/sim/postEE', 'Run3_2022/data'], ['Run3_2023/sim/preBPix', 'Run3_2023/sim/postBPix', 'Run3_2023/data'], ['Run3_2024/sim', 'Run3_2024/data'], ['Run3_2025/sim', 'Run3_2025/data']]):
    #     year_mask = test['AUX_sample_era'].isin(year_tags)
    #     print(sample, ' ', year, ' ', len(test.loc[np.logical_and(sample_mask, year_mask)]))
    print('num events of ', sample, ' ', 'all years', ' ', len(test.loc[sample_mask]))
    print('-'*60)

In [ ]:
import awkward as ak
import numba as nb

@nb.njit
def issubset(subset_candidate_arr, superset_candidate_arr, nonsubset_builder):
    nonsubset_builder.begin_list()
    for sub_elem in subset_candidate_arr:
        found = False
        for super_elem in superset_candidate_arr:
            found = found or (sub_elem == super_elem)
        nonsubset_builder.append(sub_elem * found)
    nonsubset_builder.end_list()
    return nonsubset_builder

In [ ]:
nonsubset_events = issubset(test['lumievent'].to_numpy(), manos_sample['lumievent'].to_numpy(), ak.ArrayBuilder()).snapshot()
# nonsubset_events